# 01 — Baseline: ince ayar öncesi Gemma 4 E4B

Test kümesi (720 firma-ay, `reference_seed42`) için iki varyantın yorumlarını üretir (CLAUDE.md, Bölüm 7):

| Varyant | Girdi | Karşılaştırmadaki yeri |
|---|---|---|
| `ohne_kontext` | FACTS; WISSEN ve KONTEXTNOTIZEN „Keine.“ | Çıplak model |
| `rag_k8` | FACTS + WISSEN + Firmenprofil + top-8 not | Yalnızca RAG |

- İstemler repoda hazırlandı (istem 1.1, `gpu/data/test_*.jsonl.gz`). GPU tarafında facts ve embedding hesaplanmaz.
- Ayarlar deneme turuyla aynı: `temperature 0.2`, `seed 42`, en fazla 1.500 yeni token. Model 4-bit (QLoRA ile aynı taban).
- Çıktı: `/kaggle/working/outputs/baseline_<varyant>.jsonl`. Yarıda kalırsa yeniden çalıştırınca kaldığı yerden devam eder.
- Kullanım: `gpu/README.md`. Önce `MODE = "smoke"` ile 2 istem; sorun yoksa `MODE = "full"`.

In [ ]:
# Unsloth: Gemma 4 için T4'te float16 düzeltmeleri ve ince ayarla aynı sohbet şablonu
!pip install -q --upgrade unsloth

In [ ]:
MODE = "smoke"                              # "smoke": varyant başına 2 istem · "full": bütün test kümesi
VARIANTS = ["ohne_kontext", "rag_k8"]       # çıplak model, yalnızca RAG
MODEL = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"   # önceden 4-bit (unsloth/gemma-4-E4B-it ile aynı model)
MAX_SEQ = 10240                             # en uzun istem ~7.500 token + 1.500 yeni token
MAX_NEW_TOKENS = 1500
GEN = dict(do_sample=True, temperature=0.2, top_p=0.95, top_k=64)
SEED = 42
BATCH = {"ohne_kontext": 8, "rag_k8": 4, "rag_alle": 4}    # bellek yetmezse kendiliğinden yarıya iner

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    # Kaggle'da iki T4 görünür; Unsloth tek GPU kullanır

import glob, gzip, json, time, traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

def find(stem):
    """Veri dosyası: .jsonl.gz ya da (Kaggle yüklerken sıkıştırmayı açtıysa) .jsonl."""
    for pattern in (f"/kaggle/input/**/{stem}.jsonl.gz", f"/kaggle/input/**/{stem}.jsonl"):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    seen = sorted(glob.glob("/kaggle/input/**/*", recursive=True))[:30]
    raise FileNotFoundError(f"{stem} bulunamadı. Sağ panel → Input → Add Input → veri setini ekle. "
                            f"/kaggle/input içinde görülen: {seen or 'hiçbir şey'}")

def load(variant):
    path = find(f"test_{variant}")
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

OUT = Path("/kaggle/working/outputs")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / f"log_{MODE}.txt"

def log(*parts):
    """Ekrana ve günlük dosyasına (Output'tan indirilebilir; oturum kapansa da kalır)."""
    line = " ".join(str(p) for p in parts)
    print(line, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"{datetime.now(timezone.utc).isoformat(timespec='seconds')} {line}\n")

log(f"Başlangıç · MODE={MODE} · torch {torch.__version__} · GPU: "
    f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")
log("İstemler:", {v: len(load(v)) for v in VARIANTS})

In [ ]:
from unsloth import FastModel

t0 = time.time()
try:
    model, tokenizer = FastModel.from_pretrained(MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
                                                 full_finetuning=False)
    try:
        FastModel.for_inference(model)
    except AttributeError:
        model.eval()
except Exception:
    log("MODEL YÜKLENEMEDİ:\n" + traceback.format_exc())
    raise
tok = getattr(tokenizer, "tokenizer", tokenizer)    # çok kipli modelde metin tokenizer'ı
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
free, total = torch.cuda.mem_get_info()
sizes = Counter()
for prm in model.parameters():
    sizes[str(prm.dtype)] += prm.numel()
log(f"Model yüklendi: {time.time() - t0:.0f} s · GPU belleği {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB · "
    f"parametreler: " + ", ".join(f"{k} {v / 1e9:.2f} Mrd." for k, v in sizes.most_common()))

In [ ]:
def render(messages):
    """Modelin kendi sohbet şablonu, düşünme kipi kapalı."""
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                             enable_thinking=False)
    except Exception:
        msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                             enable_thinking=False)

@torch.inference_mode()
def generate(batch, seed):
    texts = [render(r["messages"]) for r in batch]
    has_bos = bool(tok.bos_token) and texts[0].startswith(tok.bos_token)   # şablon <bos> içeriyorsa ikinci kez ekleme
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=not has_bos).to("cuda")
    torch.manual_seed(seed)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, **GEN)
    new = out[:, enc["input_ids"].shape[1]:]
    return (tok.batch_decode(new, skip_special_tokens=True),
            enc["attention_mask"].sum(dim=1).tolist(),
            (new != tok.pad_token_id).sum(dim=1).tolist())

def run(variant):
    rows = load(variant)[: 2 if MODE == "smoke" else None]
    path = OUT / f"baseline_{variant}.jsonl"
    done = set()
    if path.exists():
        with open(path, encoding="utf-8") as f:
            done = {json.loads(line)["id"] for line in f if line.strip()}
    todo = [r for r in rows if r["id"] not in done]
    todo.sort(key=lambda r: -sum(len(m["content"]) for m in r["messages"]))   # uzunlar önce: bellek sorunu erken çıkar
    bs, i = BATCH.get(variant, 4), 0
    log(f"{variant}: {len(rows)} istem, {len(done)} tamam, {len(todo)} çalışacak")
    while i < len(todo):
        batch, t0 = todo[i:i + bs], time.time()
        try:
            texts, prompt_tokens, completion_tokens = generate(batch, SEED + len(done) + i)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            log(f"  bellek yetmedi; parti boyutu → {bs}")
            continue
        dt = time.time() - t0
        with open(path, "a", encoding="utf-8") as f:
            for r, text, p, c in zip(batch, texts, prompt_tokens, completion_tokens):
                f.write(json.dumps({
                    "id": r["id"], "variant": variant, "prompt_version": r["prompt_version"], "model": MODEL,
                    "adapter": None, "text": text, "prompt_tokens": p, "completion_tokens": c,
                    "batch_size": len(batch), "batch_seconds": round(dt, 1),
                    "settings": {**GEN, "max_new_tokens": MAX_NEW_TOKENS, "seed": SEED + len(done) + i},
                    "finished": datetime.now(timezone.utc).isoformat()}, ensure_ascii=False) + "\n")
        i += len(batch)
        log(f"  [{len(done) + i}/{len(rows)}] {len(batch)} istem · {dt:.0f} s · "
            f"{sum(completion_tokens) / dt:.0f} token/s")
    return path

In [ ]:
try:
    paths = [run(v) for v in VARIANTS]
except Exception:
    log("ÜRETİM HATASI:\n" + traceback.format_exc())
    raise
if MODE == "smoke":
    for p in paths:
        with open(p, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                log(f"\n===== {rec['variant']} · {rec['id']} · girdi {rec['prompt_tokens']} · "
                    f"çıktı {rec['completion_tokens']} token · parti {rec['batch_seconds']} s\n{rec['text']}")
log("Bitti.")

**Bitince:** Sağ paneldeki **Output** bölümünden `outputs/` klasöründeki dosyaları (`log_*.txt`, `baseline_*.jsonl`) indir ve repoya yükle (`gpu/README.md`, Bölüm 5).